In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd /content/Depth-Anything-V2
!pip install -q -r metric_depth/requirements.txt ultralytics
print("Done")

In [ ]:
import os, urllib.request

CHECKPOINT_DIR = "/content/Depth-Anything-V2/metric_depth/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "depth_anything_v2_metric_hypersim_vits.pth")
URL = "https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Small/resolve/main/depth_anything_v2_metric_hypersim_vits.pth"

if not os.path.exists(CHECKPOINT_PATH):
    urllib.request.urlretrieve(URL, CHECKPOINT_PATH)

print("Size MB:", round(os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024), 2))

In [ ]:
import sys, importlib, torch

metric_path = "/content/Depth-Anything-V2/metric_depth"
if metric_path not in sys.path:
    sys.path.insert(0, metric_path)
importlib.invalidate_caches()

from depth_anything_v2.dpt import DepthAnythingV2

depth_model = DepthAnythingV2(
    encoder="vits", features=64,
    out_channels=[48, 96, 192, 384], max_depth=20
)
depth_model.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True),
    strict=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
depth_model = depth_model.to(device).eval()
print("Model loaded on", device)

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov5nu.pt")  # Using YOLOv5 Nano

print("YOLOv5 model loaded successfully!")

In [ ]:

def detect_objects(image_bgr, confidence=0.25):

    if image_bgr is None:
        raise ValueError("Input image is empty.")

    results = yolo_model.predict(
        source=image_bgr,
        conf=confidence,
        verbose=False
    )

    detections = []

    for result in results:
        for box in result.boxes:

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

            class_id = int(box.cls[0])
            label = result.names[class_id]
            score = float(box.conf[0])

            bbox = (
                int(x1),
                int(y1),
                int(x2),
                int(y2)
            )

            detections.append({
                "object": label,
                "confidence": score,
                "bbox": bbox
            })

    return detections

print("Object detection function ready!")


In [ ]:

import numpy as np

INPUT_SIZE = 256
INSET_RATIO = 0.20

def generate_depth_map(image_bgr):
    if image_bgr is None:
        raise ValueError("Image is empty.")

    depth = depth_model.infer_image(image_bgr, INPUT_SIZE)
    return np.asarray(depth, dtype=np.float32)

print("Depth model is ready")


In [ ]:

def calculate_median_depth(depth_map, bbox, inset_ratio=INSET_RATIO):

    height, width = depth_map.shape[:2]

    x1, y1, x2, y2 = map(int, bbox)

    x1 = max(0, min(x1, width))
    x2 = max(0, min(x2, width))
    y1 = max(0, min(y1, height))
    y2 = max(0, min(y2, height))

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid bounding box.")

    inset_x = int((x2 - x1) * inset_ratio)
    inset_y = int((y2 - y1) * inset_ratio)

    region = depth_map[
        y1 + inset_y:y2 - inset_y,
        x1 + inset_x:x2 - inset_x
    ]

    if region.size == 0:
        region = depth_map[y1:y2, x1:x2]

    valid = region[np.isfinite(region) & (region > 0)]

    if valid.size == 0:
        raise ValueError("No valid depth values found.")

    return float(np.median(valid))

print("Median depth function ready")


In [ ]:

SCALE_FACTOR = 0.444  # Mathematically calibrated to map 4.5m raw depth down to 2.0m real depth
OFFSET = 0.0

def calibrate_distance(raw_depth, scale=SCALE_FACTOR, offset=OFFSET):
    # The Metric Depth Anything model outputs direct distance, but requires global scaling.
    calibrated_distance = (raw_depth * scale) + offset
    return float(max(0.0, calibrated_distance))

print("Calibration loaded successfully!")
print("Scale Factor:", SCALE_FACTOR)
print("Offset:", OFFSET)


In [ ]:

def fit_calibration(raw_depths, actual_distances):
    """
    Calculate calibration slope and intercept
    from raw depth values and actual measured distances.
    """
    import numpy as np
    raw_depths = np.asarray(raw_depths, dtype=np.float64)
    actual_distances = np.asarray(actual_distances, dtype=np.float64)

    if len(raw_depths) != len(actual_distances):
        raise ValueError("Both lists must have the same length.")

    if len(raw_depths) < 2:
        raise ValueError("At least two calibration samples are required.")

    slope, intercept = np.polyfit(
        raw_depths, # Direct linear fit
        actual_distances,
        1
    )

    print("Slope:", slope)
    print("Intercept:", intercept)

    return float(slope), float(intercept)

print("Calibration fitting function is ready!")


In [ ]:

def estimate_distance(depth_map, bbox, scale, offset):
    raw_depth = calculate_median_depth(depth_map, bbox)

    calibrated = calibrate_distance(
        raw_depth,
        scale,
        offset
    )

    return {
        "raw_depth_m": raw_depth,
        "calibrated_distance_m": calibrated
    }


def process_detection(image_bgr, bbox, scale, offset):
    depth_map = generate_depth_map(image_bgr)

    return estimate_distance(
        depth_map,
        bbox,
        scale,
        offset
    )


print("Distance estimation pipeline is ready!")


In [ ]:
VEHICLE_CLASSES = {"car", "truck", "bus", "motorcycle", "bicycle"}

def classify_proximity(label, distance_m):
    """
    Categorize distance into 'near', 'medium', or 'far' thresholds
    for vehicles/cars vs. indoor objects, and flag the 'near' danger zone.
    """
    is_vehicle = str(label).lower() in VEHICLE_CLASSES
    if is_vehicle:
        # Outdoor / car thresholds (vehicles move fast and require earlier alerts)
        if distance_m < 3.0:
            level = "near"
        elif distance_m <= 7.0:
            level = "medium"
        else:
            level = "far"
    else:
        # Indoor object thresholds
        if distance_m < 1.2:
            level = "near"
        elif distance_m <= 2.5:
            level = "medium"
        else:
            level = "far"

    danger_zone = (level == "near")
    return level, danger_zone


def detect_objects_with_distance(image_bgr):

    detections = detect_objects(image_bgr)

    depth_map = generate_depth_map(image_bgr)

    output = []

    for detection in detections:

        bbox = detection["bbox"]

        distance_result = estimate_distance(
            depth_map,
            bbox,
            SCALE_FACTOR,
            OFFSET
        )

        dist_m = distance_result["calibrated_distance_m"]
        proximity_level, danger_zone = classify_proximity(detection["object"], dist_m)

        output.append({
            "object": detection["object"],
            "confidence": detection["confidence"],
            "bbox": bbox,
            "raw_depth_m": distance_result["raw_depth_m"],
            "calibrated_distance_m": dist_m,
            "proximity_level": proximity_level,
            "danger_zone": danger_zone,
            "vibrate": danger_zone
        })

    return output

print("YOLO + Depth + Proximity & Danger Zone integration ready!")

In [ ]:
import cv2

def draw_detections(image_bgr, detections):

    output_image = image_bgr.copy()

    for item in detections:

        x1, y1, x2, y2 = item["bbox"]

        label = item["object"]
        distance = item["calibrated_distance_m"]
        proximity = item.get("proximity_level", "")
        danger = item.get("danger_zone", False)

        color = (0, 0, 255) if danger else (0, 255, 0)
        tag = f" [{proximity.upper()}{' - DANGER!' if danger else ''}]" if proximity else ""
        text = f"{label}: {distance:.2f} m{tag}"

        cv2.rectangle(
            output_image,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        cv2.putText(
            output_image,
            text,
            (x1, max(25, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

    return output_image

print("Bounding box visualization ready!")

In [ ]:
import cv2
import numpy as np

# --- Feature 4: Dark / Blocked Camera Detection ---
def check_camera_visibility(frame, dark_thresh=35.0, blur_laplacian_thresh=15.0, edge_std_thresh=12.0):
    """
    Check frame brightness and texture variance to detect if the camera is
    in the dark or physically blocked/covered.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mean_brightness = float(np.mean(gray))
    std_brightness = float(np.std(gray))
    laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    is_dark = mean_brightness < dark_thresh
    is_blocked = (std_brightness < edge_std_thresh) and (laplacian_var < blur_laplacian_thresh)

    status = "ok"
    warning_message = None
    if is_dark:
        status = "dark"
        warning_message = "Camera environment is too dark."
    elif is_blocked:
        status = "blocked"
        warning_message = "Camera lens appears to be blocked or covered."

    return {
        "status": status,
        "is_dark": is_dark,
        "is_blocked": is_blocked,
        "mean_brightness": round(mean_brightness, 2),
        "contrast_std": round(std_brightness, 2),
        "warning_message": warning_message
    }


def process_frame(frame):
    """
    Process one frame received from the phone camera.
    Input: OpenCV BGR frame.
    Output: annotated frame and detections.
    """

    if frame is None:
        raise ValueError("Input frame is empty.")

    detections = detect_objects_with_distance(frame)

    output_frame = draw_detections(
        frame,
        detections
    )

    return output_frame, detections


print("process_frame and visibility check functions are ready!")

In [ ]:
import time
import cv2

class RealTimeProcessor:

    def __init__(self):
        self.fps = 0.0
        self.alpha = 0.2
        self.total_frames = 0

    def process(self, frame):

        if frame is None:
            raise ValueError("Empty frame received")

        start_time = time.perf_counter()

        output_frame, detections = process_frame(frame)

        processing_time = time.perf_counter() - start_time

        self.total_frames += 1

        current_fps = (
            1.0 / processing_time
            if processing_time > 0 else 0
        )

        if self.fps == 0:
            self.fps = current_fps
        else:
            self.fps = (
                self.alpha * current_fps
                + (1 - self.alpha) * self.fps
            )

        cv2.putText(
            output_frame,
            f"Processing FPS: {self.fps:.2f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

        return output_frame, detections, {
            "processing_time": processing_time,
            "processing_fps": self.fps,
            "total_frames": self.total_frames
        }


realtime_processor = RealTimeProcessor()

print("RealTimeProcessor is ready!")

## Real-time connection with the Android app
This notebook does **not** turn on the phone camera itself. The Flutter app opens the Android camera and continuously uploads frames to the API below. This notebook receives each frame, runs YOLO + Depth Anything + calibration, and returns detections, distance, annotated image, and FPS.
Run the cells from top to bottom. Keep this Colab session running while the app is connected. The public Cloudflare URL printed in the final cell is temporary and must be placed in the Flutter app by the teammate handling Flutter.


In [ ]:

!pip install -q fastapi uvicorn python-multipart nest-asyncio


In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov5nu.pt")

exported_model = yolo_model.export(
    format="tflite",
    imgsz=320,
    batch=1
)

print("YOLOv5 Android export completed!")
print("Exported model:", exported_model)

In [ ]:
import time
import threading

processing_lock = threading.Lock()

total_frames_processed = 0
total_processing_time = 0.0

def process_received_frame(frame):

    global total_frames_processed
    global total_processing_time

    if frame is None:
        raise ValueError("Empty frame received")

    if not processing_lock.acquire(blocking=False):
        return None

    try:
        start_time = time.perf_counter()

        # Process every received frame
        output_frame, detections = process_frame(frame)

        elapsed = time.perf_counter() - start_time

        total_frames_processed += 1
        total_processing_time += elapsed

        average_fps = (
            total_frames_processed / total_processing_time
            if total_processing_time > 0 else 0
        )

        processing_fps = (
            1 / elapsed if elapsed > 0 else 0
        )

        return {
            "output_frame": output_frame,
            "detections": detections,
            "processing_time": elapsed,
            "processing_fps": processing_fps,
            "average_fps": average_fps,
            "total_frames_processed": total_frames_processed
        }
    finally:
        processing_lock.release()

print("Continuous frame processing is ready!")

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np

print("All libraries imported successfully!")

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np

# Prepare the loaded model
depth_model = depth_model.cpu().eval()

class ExportModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, image):
        return self.model(image)

export_model = ExportModel(depth_model).eval()

# Input dimensions
dummy_input = torch.randn(1, 3, 686, 518)

ONNX_PATH = "/content/depth_anything_metric.onnx"

with torch.no_grad():
    torch.onnx.export(
        export_model,
        dummy_input,
        ONNX_PATH,
        input_names=["image"],
        output_names=["depth"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False
    )

# Validate ONNX model
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

# Test inference
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

test_input = np.random.randn(
    1, 3, 686, 518
).astype(np.float32)

result = session.run(
    ["depth"],
    {"image": test_input}
)

print("ONNX export successful!")
print("Output shape:", result[0].shape)
print("Model size MB:", round(
    __import__("os").path.getsize(ONNX_PATH) / (1024 * 1024), 2
))

In [ ]:
# Compare PyTorch and ONNX outputs using the same input

test_input = np.random.randn(
    1, 3, 686, 518
).astype(np.float32)

with torch.no_grad():
    torch_result = depth_model(
        torch.from_numpy(test_input)
    ).cpu().numpy()

onnx_result = session.run(
    ["depth"],
    {"image": test_input}
)[0]

max_diff = np.max(
    np.abs(torch_result - onnx_result)
)

mean_diff = np.mean(
    np.abs(torch_result - onnx_result)
)

print("PyTorch shape:", torch_result.shape)
print("ONNX shape:", onnx_result.shape)
print("Maximum difference:", max_diff)
print("Mean difference:", mean_diff)

if np.allclose(
    torch_result,
    onnx_result,
    atol=1e-3,
    rtol=1e-3
):
    print("SUCCESS: ONNX matches PyTorch!")
else:
    print("WARNING: Outputs are different!")

# Restore the model to the selected runtime device for live frame processing.
depth_model = depth_model.to(device).eval()
print("Depth model restored to", device)


In [ ]:
import time
import threading
from collections import deque

processing_lock = threading.Lock()
total_frames_processed = 0
total_processing_time = 0.0
realtime_processor = RealTimeProcessor()

# --- Feature 3: Multi-Frame Tracking & Announcement Rules ---
class ObjectAnnouncementTracker:
    """
    Tracks objects across frames to:
    1. Confirm an object only if seen in at least 2 out of the last 3 frames.
    2. Enforce a 5-second cooldown per object class before repeating an announcement.
    3. Re-announce earlier if a brand-new object enters or an existing object moves
       significantly closer (by >= 0.4m) or enters the 'near' danger zone.
    """
    def __init__(self, window_size=3, min_hits=2, cooldown_sec=5.0, closer_delta_m=0.40):
        self.window_size = window_size
        self.min_hits = min_hits
        self.cooldown_sec = cooldown_sec
        self.closer_delta_m = closer_delta_m
        self.frame_history = deque(maxlen=window_size)
        self.last_announced = {}  # label -> {"time": float, "distance": float, "proximity": str}

    def update_and_filter(self, detections):
        now = time.time()
        current_labels = {d["object"] for d in detections}
        self.frame_history.append(current_labels)

        confirmed_detections = []
        announcements = []

        for det in detections:
            label = det["object"]
            dist = det["calibrated_distance_m"]
            prox = det.get("proximity_level", "far")
            danger = det.get("danger_zone", False)

            # Rule 1: Confirm object if present in at least 2 out of last 3 frames
            hits = sum(1 for frame_set in self.frame_history if label in frame_set)
            is_confirmed = hits >= self.min_hits
            det["confirmed"] = is_confirmed

            if not is_confirmed:
                det["should_announce"] = False
                continue

            confirmed_detections.append(det)

            # Rule 2 & 3: Cooldown (5s) + New or Significantly Closer Object override
            prev = self.last_announced.get(label)
            should_announce = False
            reason = ""

            if prev is None:
                should_announce = True
                reason = "new_object"
            else:
                elapsed = now - prev["time"]
                moved_closer = (prev["distance"] - dist) >= self.closer_delta_m
                entered_danger = danger and (prev["proximity"] != "near")

                if entered_danger:
                    should_announce = True
                    reason = "entered_danger_zone"
                elif moved_closer and elapsed >= 1.5:
                    should_announce = True
                    reason = "object_closer"
                elif elapsed >= self.cooldown_sec:
                    should_announce = True
                    reason = "cooldown_elapsed"

            det["should_announce"] = should_announce
            if should_announce:
                self.last_announced[label] = {
                    "time": now,
                    "distance": dist,
                    "proximity": prox
                }
                announcements.append({
                    "object": label,
                    "distance_m": round(dist, 2),
                    "proximity_level": prox,
                    "danger_zone": danger,
                    "reason": reason,
                    "speech_text": f"{'Warning! ' if danger else ''}{label} {prox}, {dist:.1f} meters"
                })

        return confirmed_detections, announcements

announcement_tracker = ObjectAnnouncementTracker(
    window_size=3, min_hits=2, cooldown_sec=5.0, closer_delta_m=0.40
)


def process_received_frame(frame):
    """Process one BGR frame received from the Android Flutter app."""
    global total_frames_processed, total_processing_time
    if frame is None or not isinstance(frame, np.ndarray):
        raise ValueError("Empty or invalid frame received")

    if not processing_lock.acquire(blocking=False):
        return None

    try:
        start_time = time.perf_counter()

        # 1. Check for dark or blocked camera first
        visibility = check_camera_visibility(frame)

        # 2. Run object detection, metric depth, proximity levels & danger zone
        output_frame, detections, stats = realtime_processor.process(frame)

        # Convert NumPy/native values into JSON-safe Python values.
        safe_detections = []
        for item in detections:
            safe_detections.append({
                "object": str(item["object"]),
                "confidence": float(item["confidence"]),
                "bbox": [int(v) for v in item["bbox"]],
                "raw_depth_m": float(item["raw_depth_m"]),
                "calibrated_distance_m": float(item["calibrated_distance_m"]),
                "proximity_level": str(item.get("proximity_level", "far")),
                "danger_zone": bool(item.get("danger_zone", False)),
                "vibrate": bool(item.get("vibrate", False))
            })

        # 3. Apply 2-of-3 confirmation, 5s cooldown, and new/closer announcement rules
        confirmed_detections, announcements = announcement_tracker.update_and_filter(safe_detections)

        elapsed = time.perf_counter() - start_time
        total_frames_processed += 1
        total_processing_time += elapsed
        average_fps = (total_frames_processed / total_processing_time
                       if total_processing_time > 0 else 0.0)

        # Trigger vibration flag if any confirmed object is in the danger zone
        trigger_vibration = any(d["danger_zone"] for d in confirmed_detections)

        return {
            "output_frame": output_frame,
            "detections": safe_detections,
            "confirmed_detections": confirmed_detections,
            "announcements": announcements,
            "trigger_vibration": trigger_vibration,
            "camera_visibility": visibility,
            "processing_time": float(elapsed),
            "processing_fps": float(stats["processing_fps"]),
            "average_fps": float(average_fps),
            "total_frames_processed": int(total_frames_processed)
        }
    finally:
        processing_lock.release()

print("Frame receiver with tracking & visibility check is ready")

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from fastapi.concurrency import run_in_threadpool
from fastapi.middleware.cors import CORSMiddleware
import base64
import cv2
import numpy as np

app = FastAPI(title="Basirati Real-Time Processing API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Restrict to your app's origin/domain for production.
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app.get("/health")
async def health():
    return {"status": "ok", "message": "Basirati frame API is running"}

@app.post("/process-frame")
async def process_camera_frame(file: UploadFile = File(...)):
    image_bytes = await file.read()
    if not image_bytes:
        raise HTTPException(status_code=400, detail="Empty image upload")

    image_array = np.frombuffer(image_bytes, dtype=np.uint8)
    frame = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    if frame is None:
        raise HTTPException(status_code=400, detail="Invalid image frame")

    try:
        # Run the CPU/GPU-heavy model inference outside the async event loop.
        result = await run_in_threadpool(process_received_frame, frame)
        
        # If the lock is active, drop the frame to prevent API lag
        if result is None:
            return JSONResponse(
                status_code=200,
                content={"status": "dropped", "message": "Frame dropped to prevent lag"}
            )
            
        success, encoded = cv2.imencode(
            ".jpg", result["output_frame"]
        )
        if not success:
            raise ValueError("Could not encode processed frame")

        return {
            "status": "processed",
            "detections": result["detections"],
            "confirmed_detections": result["confirmed_detections"],
            "announcements": result["announcements"],
            "trigger_vibration": result["trigger_vibration"],
            "camera_visibility": result["camera_visibility"],
            "output_frame": base64.b64encode(encoded.tobytes()).decode("utf-8"),
            "processing_time": result["processing_time"],
            "processing_fps": result["processing_fps"],
            "average_fps": result["average_fps"],
            "total_frames_processed": result["total_frames_processed"]
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

print("Basirati API routes are ready with anti-lag measures")

In [ ]:
import os, urllib.request, threading, time, uvicorn, nest_asyncio, re

nest_asyncio.apply()

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

if "api_server_thread" not in globals() or not api_server_thread.is_alive():
    def _run_api():
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
    api_server_thread = threading.Thread(target=_run_api, daemon=True)
    api_server_thread.start()
    time.sleep(3)

try:
    with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as response:
        print("Local API:", response.read().decode())
except Exception as exc:
    raise RuntimeError("FastAPI did not start.") from exc

# Safely start Cloudflare in the background and read from a log file
!nohup /usr/local/bin/cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &

print("Waiting 6 seconds for Cloudflare to generate the URL...")
time.sleep(6)

if os.path.exists("cloudflared.log"):
    with open("cloudflared.log", "r") as f:
        log_content = f.read()

    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)

    if match:
        public_url = match.group(0)
        print("\n✅ API PUBLIC URL:", public_url)
        print("Health check:", public_url + "/health")
        print("Flutter frame endpoint:", public_url + "/process-frame")
    else:
        print("⚠️ Could not find Cloudflare URL. Here is the exact log output to see what went wrong:")
        print("--- START LOG ---")
        print(log_content)
        print("--- END LOG ---")
else:
    print("Error: Cloudflare log file was not created.")
